# 04 — Build the Excel quality-review workbook

This optional notebook creates one concise Excel workbook from notebook 03 exports. It is for human review and reconciliation, not for modifying the shared data model.

In [ ]:
# Import workbook tools, locate the project, and check that notebook 03 has finished.
from pathlib import Path
import json
import pandas as pd
import yaml
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.utils import get_column_letter


def find_project_root(start: Path) -> Path:
    """Find the project root from the active Jupyter working directory."""
    for candidate in [start, *start.parents]:
        if (candidate / "notebooks" / "data").exists():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the project folder.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
EXPORT_DIR = PROJECT_ROOT / "notebooks" / "data" / "exports"
SUMMARY_PATH = PROJECT_ROOT / "notebooks" / "data" / "qa_summary.json"
WORKBOOK_PATH = PROJECT_ROOT / "excel" / "job_market_quality_review.xlsx"
required_exports = ["fact_job_posting", "fact_job_quality", "fact_job_flag", "dim_company", "dim_role", "dim_location"]
missing_exports = [name for name in required_exports if not (EXPORT_DIR / f"{name}.csv").exists()]
if missing_exports or not SUMMARY_PATH.exists():
    raise FileNotFoundError("Run notebook 03 before building the Excel workbook.")

In [ ]:
# Load only the generated exports required for a reviewer-friendly sample.
with (PROJECT_ROOT / "notebooks" / "analysis_settings.yaml").open(encoding="utf-8") as file:
    settings = yaml.safe_load(file)
with SUMMARY_PATH.open(encoding="utf-8") as file:
    summary = json.load(file)

postings = pd.read_csv(EXPORT_DIR / "fact_job_posting.csv", low_memory=False)
quality = pd.read_csv(EXPORT_DIR / "fact_job_quality.csv", low_memory=False)
flags = pd.read_csv(EXPORT_DIR / "fact_job_flag.csv", low_memory=False)
companies = pd.read_csv(EXPORT_DIR / "dim_company.csv", low_memory=False)
roles = pd.read_csv(EXPORT_DIR / "dim_role.csv", low_memory=False)
locations = pd.read_csv(EXPORT_DIR / "dim_location.csv", low_memory=False)

# Join descriptive names to the fact table so a reviewer does not need to look up keys.
qa_base = (
    postings
    .merge(quality[["job_key", "jd_quality_score", "quality_band", "flag_count"]], on="job_key", how="left")
    .merge(companies[["company_key", "company_name"]], on="company_key", how="left")
    .merge(roles[["role_key", "canonical_role_name"]], on="role_key", how="left")
    .merge(locations[["location_key", "location_normalized"]], on="location_key", how="left")
)
flag_text = flags.groupby("job_key")["rule_code"].agg(lambda values: "; ".join(sorted(set(values)))).rename("rule_codes") if not flags.empty else pd.Series(dtype="object", name="rule_codes")
qa_base = qa_base.merge(flag_text, on="job_key", how="left")
qa_base["description_excerpt"] = qa_base["description_clean"].fillna("").str[:1000]
qa_sample = qa_base.sample(min(200, len(qa_base)), random_state=42).sort_values(["quality_band", "jd_quality_score"])
print(f"QA sample rows: {len(qa_sample):,}")

In [ ]:
# Create small helper functions so every sheet has consistent headers and readable column widths.
HEADER_FILL = PatternFill("solid", fgColor="1F4E78")
HEADER_FONT = Font(color="FFFFFF", bold=True)
INPUT_FILL = PatternFill("solid", fgColor="FFF2CC")


def safe_value(value):
    """Turn pandas missing values into Excel blanks while keeping numbers and text intact."""
    if pd.isna(value):
        return None
    return "'" + value if isinstance(value, str) and value.startswith(("=", "+", "-", "@")) else value


def write_table(sheet, start_row, headers, rows):
    """Write a header plus data rows and return the final written row number."""
    for column, header in enumerate(headers, start=1):
        cell = sheet.cell(start_row, column, header)
        cell.fill = HEADER_FILL
        cell.font = HEADER_FONT
    final_row = start_row
    for row_number, row_values in enumerate(rows, start=start_row + 1):
        for column, value in enumerate(row_values, start=1):
            sheet.cell(row_number, column, safe_value(value))
        final_row = row_number
    return final_row


def fit_columns(sheet, maximum_width=45):
    """Fit visible content without creating extremely wide description columns."""
    for column_cells in sheet.columns:
        letter = get_column_letter(column_cells[0].column)
        longest = max(len(str(cell.value or "")) for cell in column_cells)
        sheet.column_dimensions[letter].width = min(max(longest + 2, 12), maximum_width)

In [ ]:
# Build the Overview sheet from the same notebook-03 summary used for cross-tool reconciliation.
workbook = Workbook()
overview = workbook.active
overview.title = "Overview"
overview["A1"] = "Job Market Quality Review"
overview["A1"].font = Font(size=16, bold=True, color="1F2937")
overview["A2"] = "Historical LinkedIn posting snapshot. Use as a quality-review aid, not a current job-market claim."

overview_rows = [
    ("Posting observations", summary["rows_silver"], "Rows in fact_job_posting"),
    ("Distinct canonical jobs", summary["distinct_jobs"], "Rows in dim_job"),
    ("Average JD quality", round(float(quality["jd_quality_score"].mean()), 2), "Mean score from fact_job_quality"),
    ("Salary numeric coverage", summary["salary_numeric_coverage_pct"] / 100, "Postings with a numeric salary value"),
    ("Entry-level jobs", summary["entry_jobs_count"], "Titles classified as entry"),
    ("Entry jobs requiring 2+ years", summary["entry_jobs_requiring_2plus_years"], "Records flagged for human review"),
    ("Exact repost events", summary["repost_events_count"], "Exact duplicate clusters after the first observation"),
    ("Rule flags", summary["total_flags_generated"], "All transparent rule flags")
]
write_table(overview, 4, ["Metric", "Value", "Definition"], overview_rows)
overview["B8"].number_format = "0.0%"
overview.freeze_panes = "A5"

In [ ]:
# Create a reviewer table with two editable columns and a controlled decision list.
qa_sheet = workbook.create_sheet("QA Sample")
qa_sheet["A1"] = "Human review sample"
qa_sheet["A1"].font = Font(size=16, bold=True, color="1F2937")
qa_sheet["A2"] = "Review the record and use the two yellow columns to document your decision."
qa_columns = [
    "source_job_id", "company_name", "title_raw", "canonical_role_name", "location_normalized",
    "workplace_type", "experience_min_years", "salary_min", "salary_max", "salary_currency",
    "salary_period", "salary_median", "jd_quality_score", "quality_band", "flag_count", "rule_codes",
    "description_excerpt", "reviewer_decision", "reviewer_note"
]
qa_export = qa_sample.reindex(columns=[column for column in qa_columns if column not in {"reviewer_decision", "reviewer_note"}]).copy()
qa_export["reviewer_decision"] = ""
qa_export["reviewer_note"] = ""
write_table(qa_sheet, 4, qa_columns, qa_export.itertuples(index=False, name=None))
qa_sheet.freeze_panes = "A5"

# Highlight only the human-input columns and constrain reviewer decisions to three clear values.
for row in range(5, 5 + len(qa_export)):
    qa_sheet.cell(row, 18).fill = INPUT_FILL
    qa_sheet.cell(row, 19).fill = INPUT_FILL
validation = DataValidation(type="list", formula1="\"Confirm,Needs Review,Not Applicable\"", allow_blank=True)
qa_sheet.add_data_validation(validation)
validation.add(f"R5:R{4 + len(qa_export)}")

In [ ]:
# Add Checks and Source sheets so the workbook remains traceable to the notebook run.
checks = workbook.create_sheet("Checks")
checks["A1"] = "Reconciliation checks"
checks["A1"].font = Font(size=16, bold=True, color="1F2937")
check_rows = [
    ("Posting observations", summary["rows_silver"], len(postings), summary["rows_silver"] - len(postings)),
    ("Distinct canonical jobs", summary["distinct_jobs"], int(quality["job_key"].nunique()), summary["distinct_jobs"] - int(quality["job_key"].nunique())),
    ("Rule flags", summary["total_flags_generated"], len(flags), summary["total_flags_generated"] - len(flags))
]
write_table(checks, 4, ["Check", "Notebook value", "Export value", "Variance", "Result"], check_rows)
for row in range(5, 8):
    checks.cell(row, 4, f"=B{row}-C{row}")
    checks.cell(row, 5, f'=IF(D{row}=0,"Match","Review")')
checks.freeze_panes = "A5"

source = workbook.create_sheet("Source")
source["A1"] = "Source and run settings"
source["A1"].font = Font(size=16, bold=True, color="1F2937")
source_rows = [
    ("Source file", summary["source_path"]),
    ("Source name", settings["source"]["name"]),
    ("Historical snapshot date", summary["snapshot_date"]),
    ("Row limit", str(summary["row_limit"])),
    ("Score version", summary["score_version"]),
    ("Licence note", settings["source"]["licence_note"])
]
write_table(source, 4, ["Setting", "Value"], source_rows)
source.freeze_panes = "A5"

In [ ]:
# Apply finishing touches and save the single generated workbook.
for sheet in workbook.worksheets:
    fit_columns(sheet)
    sheet.auto_filter.ref = f"A4:{get_column_letter(sheet.max_column)}{sheet.max_row}"
qa_sheet.column_dimensions["Q"].width = 80
from openpyxl.styles import Alignment
for row in qa_sheet.iter_rows(min_row=5):
    row[16].alignment = Alignment(wrap_text=True, vertical="top")
    qa_sheet.row_dimensions[row[0].row].height = 75

WORKBOOK_PATH.parent.mkdir(parents=True, exist_ok=True)
workbook.save(WORKBOOK_PATH)
print(f"Created {WORKBOOK_PATH.relative_to(PROJECT_ROOT)}")
print("Open the workbook in Excel and use the yellow columns in QA Sample for manual review.")